# 01 - Structural VAR: Cholesky and AB Model Identification

This notebook covers **Structural VAR (SVAR)** models with two identification schemes:

1. **Cholesky decomposition** (recursive identification)
2. **AB model** with explicit short-run restrictions

## Topics covered

- From reduced form to structural form
- The identification problem in VAR models
- Cholesky (triangular) identification
- The AB model: $Au_t = Be_t$
- Structural Impulse Response Functions (IRF)
- Forecast Error Variance Decomposition (FEVD) under structural shocks
- Exercises: identifying monetary policy shocks

---

## From Reduced Form to Structural Form

A **reduced-form VAR(p)** is:

$$Y_t = c + A_1 Y_{t-1} + A_2 Y_{t-2} + \cdots + A_p Y_{t-p} + u_t, \quad u_t \sim N(0, \Sigma_u)$$

The residuals $u_t$ are **correlated** across equations. They are a *mixture* of the
underlying **structural shocks** $\varepsilon_t$, which are economically meaningful and
mutually uncorrelated.

The **structural form** links the two:

$$u_t = B_0^{-1} \varepsilon_t, \quad \varepsilon_t \sim N(0, I_K)$$

where $B_0^{-1}$ is the **structural impact matrix**. This implies:

$$\Sigma_u = B_0^{-1} (B_0^{-1})'$$

The challenge: $\Sigma_u$ has only $K(K+1)/2$ unique elements, but $B_0^{-1}$ has $K^2$
free parameters. We need **at least** $K(K-1)/2$ additional restrictions to achieve
**just-identification**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os

from chronobox.models import VAR, SVAR

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_structural_irf, plot_irf_comparison

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.facecolor"] = "white"
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Loading the US Macro Dataset

We use quarterly US macroeconomic data with 4 variables:

| Variable | Description |
|----------|-------------|
| `gdp` | Real GDP growth (annualized %) |
| `inflation` | CPI inflation rate (%) |
| `fed_funds` | Federal Funds Rate (%) |
| `unemployment` | Unemployment rate (%) |

The **ordering matters** for Cholesky identification. Our baseline ordering reflects
a standard macroeconomic timing assumption:

1. **GDP** - slow-moving, does not respond within the quarter to financial variables
2. **Inflation** - responds to output gap but not to policy within the quarter
3. **Fed Funds** - the central bank observes GDP and inflation before setting the rate
4. **Unemployment** - responds to all other variables contemporaneously

This is the classic **Christiano-Eichenbaum-Evans (1999)** ordering for monetary policy analysis.

In [ ]:
# Load US macro quarterly dataset
data_path = os.path.join("..", "data", "us_macro_quarterly.csv")
df = pd.read_csv(data_path, parse_dates=["date"])
df.set_index("date", inplace=True)

var_names = ["gdp", "inflation", "fed_funds", "unemployment"]
endog = df[var_names].values

print(f"Dataset shape: {df.shape}")
print(f"Period: {df.index[0].strftime('%Y-Q%q')} to {df.index[-1].strftime('%Y-Q%q')}" 
      if hasattr(df.index[0], 'quarter') else f"Period: {df.index[0]} to {df.index[-1]}")
print(f"\nDescriptive statistics:")
df[var_names].describe().round(3)

In [ ]:
# Visualize the data
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
titles = ["GDP Growth (%)", "Inflation (%)", "Fed Funds Rate (%)", "Unemployment Rate (%)"]
colors = ["steelblue", "firebrick", "darkgreen", "darkorange"]

for i, (ax, name, title, color) in enumerate(zip(axes.flat, var_names, titles, colors)):
    ax.plot(df.index, df[name], color=color, linewidth=1.2)
    ax.set_title(title, fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color="black", linewidth=0.5)

plt.suptitle("US Macro Variables (Quarterly)", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Fitting the Reduced-Form VAR

Before structural identification, we estimate a standard **reduced-form VAR(4)**. 
The choice of 4 lags is standard for quarterly data (one year of dynamics).

The reduced-form VAR captures the joint dynamics but **cannot distinguish** between
different structural shocks — the residuals $u_t$ are correlated mixtures of structural shocks.

In [ ]:
# Fit reduced-form VAR(4)
var_model = VAR(lags=4, trend="c")
var_results = var_model.fit(endog)

print(f"VAR({var_results.lags}) fitted")
print(f"Number of observations: {var_results.n_obs}")
print(f"Number of variables: {var_results.k_vars}")
print(f"\nReduced-form residual covariance (Sigma_u):")
print(var_results.sigma_u.round(4))

# Show that residuals are correlated
corr = np.corrcoef(var_results.resid.T)
print(f"\nResidual correlation matrix:")
print(pd.DataFrame(corr, index=var_names, columns=var_names).round(3))

## 3. Cholesky Identification

The **Cholesky decomposition** is the simplest identification scheme. It decomposes:

$$\Sigma_u = PP'$$

where $P$ is a **lower triangular** matrix (the Cholesky factor). Setting $B_0^{-1} = P$ gives:

$$u_t = P \varepsilon_t$$

This imposes a **recursive (triangular) structure**:
- The 1st variable (GDP) is affected only by its own structural shock contemporaneously
- The 2nd variable (inflation) is affected by GDP and inflation shocks
- The 3rd variable (fed_funds) is affected by GDP, inflation, and monetary shocks
- The 4th variable (unemployment) is affected by all shocks

This provides exactly $K(K-1)/2 = 6$ zero restrictions — just enough for identification.

**Key insight:** Variable ordering imposes an economic **causal chain** within the period.
The first variable is the most "exogenous" and the last is the most "endogenous".

In [ ]:
# SVAR with Cholesky identification
svar_cholesky = SVAR(var_results, method="cholesky")
results_cholesky = svar_cholesky.fit()

print(f"Identification method: {results_cholesky.method}")
print(f"\nStructural impact matrix (B0_inv = Cholesky factor P):")
print(pd.DataFrame(
    results_cholesky.A0_inv.round(4),
    index=var_names, columns=[f"shock_{v}" for v in var_names]
))

# Verify: P @ P' should reconstruct Sigma_u
reconstructed = results_cholesky.A0_inv @ results_cholesky.A0_inv.T
print(f"\nReconstruction check ||Sigma_u - P@P'||: {np.linalg.norm(var_results.sigma_u - reconstructed):.2e}")

### 3.1 Structural IRFs under Cholesky

The **structural IRF** traces the effect of a **one-standard-deviation structural shock** $\varepsilon_j$ on variable $i$ at horizon $h$:

$$\text{IRF}(h, i, j) = \Phi_h B_0^{-1} e_j$$

where $\Phi_h$ are the moving average coefficients and $e_j$ is the $j$-th column selector.

Unlike reduced-form IRFs, structural IRFs have an **economic interpretation** because each
shock corresponds to a specific source of variation (e.g., monetary policy shock, demand shock).

In [ ]:
# Compute structural IRFs (20 quarters = 5 years)
irf_cholesky = results_cholesky.irf(periods=20)
print(f"IRF shape: {irf_cholesky.shape}  (periods+1, K_response, K_shock)")

# Plot all structural IRFs
shock_names = ["GDP shock", "Inflation shock", "Monetary shock", "Unemp. shock"]
fig = plot_structural_irf(
    irf_cholesky,
    variable_names=var_names,
    shock_names=shock_names,
    title="Structural IRF (Cholesky Identification)",
    figsize=(16, 12),
)
plt.savefig(os.path.join("..", "outputs", "svar_cholesky_irf_all.png"), bbox_inches="tight")
plt.show()

**Economic interpretation of the monetary policy shock (3rd column):**

- **GDP**: A contractionary monetary shock (positive fed_funds shock) should reduce GDP growth.
  The negative response typically peaks after 2-4 quarters — the "monetary transmission lag".
- **Inflation**: The "price puzzle" may appear — an initial positive response of inflation
  to a rate hike. This is a well-known empirical finding (Sims, 1992).
- **Fed Funds**: The shock itself causes an immediate jump, followed by gradual reversion.
- **Unemployment**: Should increase in response to monetary tightening, with a lag.

The Cholesky ordering assumes GDP and inflation do **not** respond within the same quarter
to a monetary shock — a reasonable assumption for quarterly data.

In [ ]:
# Focus on the monetary policy shock: response of all variables
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
horizons = np.arange(irf_cholesky.shape[0])
monetary_shock_idx = 2  # fed_funds is the 3rd variable

for i, (ax, name, title) in enumerate(zip(axes.flat, var_names, titles)):
    ax.plot(horizons, irf_cholesky[:, i, monetary_shock_idx], 
            color="darkblue", linewidth=2)
    ax.axhline(0, color="black", linewidth=0.5)
    ax.fill_between(horizons, 0, irf_cholesky[:, i, monetary_shock_idx],
                    alpha=0.15, color="blue")
    ax.set_title(f"Response of {title} to Monetary Shock", fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)

plt.suptitle("Monetary Policy Shock (Cholesky)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "svar_cholesky_monetary_shock.png"), bbox_inches="tight")
plt.show()

### 3.2 FEVD under Cholesky

The **Forecast Error Variance Decomposition** tells us what fraction of the $h$-step-ahead
forecast error of variable $i$ is explained by structural shock $j$:

$$\theta_{ij}(h) = \frac{\sum_{s=0}^{h} (\Phi_s B_0^{-1} e_j)_i^2}{\sum_{s=0}^{h} (\Phi_s \Sigma_u \Phi_s')_{ii}}$$

In [ ]:
# FEVD under Cholesky identification
fevd_cholesky = results_cholesky.fevd(periods=20)
print(f"FEVD shape: {fevd_cholesky.shape}  (periods+1, K_response, K_shock)")

# Plot FEVD as stacked area charts
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
cmap = plt.cm.Set2

for i, (ax, name) in enumerate(zip(axes.flat, var_names)):
    bottom = np.zeros(fevd_cholesky.shape[0])
    for j in range(len(var_names)):
        ax.fill_between(
            np.arange(fevd_cholesky.shape[0]),
            bottom, bottom + fevd_cholesky[:, i, j],
            label=shock_names[j], alpha=0.8, color=cmap(j / 3)
        )
        bottom += fevd_cholesky[:, i, j]
    ax.set_title(f"FEVD: {name}", fontsize=11)
    ax.set_xlabel("Horizon (quarters)")
    ax.set_ylabel("Fraction")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.2)
    if i == 0:
        ax.legend(fontsize=7, loc="center right")

plt.suptitle("Forecast Error Variance Decomposition (Cholesky)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "svar_cholesky_fevd.png"), bbox_inches="tight")
plt.show()

### 3.3 Sensitivity to Variable Ordering

A major limitation of Cholesky identification: results **change with the ordering**.
Let us verify by reversing the order to `[unemployment, fed_funds, inflation, gdp]`.

In [ ]:
# Reverse ordering: unemployment first, gdp last
reverse_names = ["unemployment", "fed_funds", "inflation", "gdp"]
endog_reversed = df[reverse_names].values

var_reversed = VAR(lags=4, trend="c")
var_res_rev = var_reversed.fit(endog_reversed)

svar_rev = SVAR(var_res_rev, method="cholesky")
res_rev = svar_rev.fit()
irf_rev = res_rev.irf(periods=20)

# Compare: response of GDP to monetary shock in both orderings
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original ordering: GDP response to fed_funds shock (shock index 2)
axes[0].plot(horizons, irf_cholesky[:, 0, 2], "b-", linewidth=2)
axes[0].axhline(0, color="k", linewidth=0.5)
axes[0].set_title("Original: [gdp, infl, ff, unemp]\nGDP response to monetary shock")
axes[0].set_xlabel("Quarters")
axes[0].grid(True, alpha=0.3)

# Reversed ordering: GDP (index 3) response to fed_funds shock (index 1)
axes[1].plot(horizons, irf_rev[:, 3, 1], "r-", linewidth=2)
axes[1].axhline(0, color="k", linewidth=0.5)
axes[1].set_title("Reversed: [unemp, ff, infl, gdp]\nGDP response to monetary shock")
axes[1].set_xlabel("Quarters")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Cholesky IRF: Sensitivity to Variable Ordering", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "svar_ordering_sensitivity.png"), bbox_inches="tight")
plt.show()

print("The IRFs differ because the Cholesky decomposition imposes different")
print("contemporaneous restrictions depending on the ordering.")

## 4. AB Model Identification

The **AB model** provides a more flexible identification scheme:

$$A u_t = B \varepsilon_t$$

where:
- $A$ is a $K \times K$ matrix capturing **contemporaneous relationships** among variables
- $B$ is a $K \times K$ matrix mapping structural shocks to innovations
- $\varepsilon_t \sim N(0, I_K)$

The identification condition is:

$$\Sigma_u = A^{-1} B B' (A^{-1})'$$

**How to specify restrictions:** Use `np.nan` for free parameters and fixed values
(e.g., 0, 1) for restrictions. The model solves for the free parameters via maximum likelihood.

### Example: Monetary Policy Model

We impose the following restrictions based on economic theory:

**A matrix** (contemporaneous effects among observables):
- GDP is not affected contemporaneously by other variables → zeros in row 1
- Inflation responds contemporaneously to GDP but not to fed_funds/unemp
- Fed Funds responds to GDP and inflation (Taylor rule)
- Unemployment responds to GDP

**B matrix** (diagonal — each structural shock has unit variance):
- Diagonal elements are free, off-diagonal elements are zero

In [ ]:
# Define A and B matrices for the AB model
# np.nan = free parameter to be estimated
# Fixed values = restrictions

K = 4

# A matrix: contemporaneous relationships
# A @ u_t = B @ eps_t
# Diagonal = 1 (normalization), off-diagonal encode restrictions
A_mat = np.array([
    [1.0,     0.0,     0.0,     0.0    ],  # GDP: no contemp. effect from others
    [np.nan,  1.0,     0.0,     0.0    ],  # Inflation: responds to GDP
    [np.nan,  np.nan,  1.0,     0.0    ],  # Fed Funds: responds to GDP, inflation
    [np.nan,  0.0,     0.0,     1.0    ],  # Unemployment: responds to GDP only
])

# B matrix: diagonal (one free param per equation)
B_mat = np.array([
    [np.nan, 0.0,    0.0,    0.0   ],
    [0.0,    np.nan, 0.0,    0.0   ],
    [0.0,    0.0,    np.nan, 0.0   ],
    [0.0,    0.0,    0.0,    np.nan],
])

print("A matrix template (nan = free parameter):")
print(A_mat)
print(f"\nFree parameters in A: {np.isnan(A_mat).sum()}")
print(f"Free parameters in B: {np.isnan(B_mat).sum()}")
print(f"Total free parameters: {np.isnan(A_mat).sum() + np.isnan(B_mat).sum()}")
print(f"Available equations (unique elements in Sigma_u): {K*(K+1)//2}")
print(f"Just-identified: {np.isnan(A_mat).sum() + np.isnan(B_mat).sum() == K*(K+1)//2}")

In [ ]:
# Fit the AB model
svar_ab = SVAR(var_results, method="ab", a_mat=A_mat, b_mat=B_mat)
results_ab = svar_ab.fit()

print(f"Identification method: {results_ab.method}")
print(f"\nEstimated A matrix:")
print(pd.DataFrame(results_ab.A0.round(4), index=var_names, columns=var_names))
print(f"\nEstimated B matrix:")
print(pd.DataFrame(results_ab.B.round(4), index=var_names, columns=var_names))

# Structural impact matrix: A^{-1} B
print(f"\nStructural impact matrix (A^{{-1}}B):")
print(pd.DataFrame(
    results_ab.A0_inv.round(4),
    index=var_names, columns=[f"shock_{v}" for v in var_names]
))

# Verify reconstruction
reconstructed_ab = results_ab.A0_inv @ results_ab.A0_inv.T
print(f"\nReconstruction check ||Sigma_u - A^{{-1}}BB'(A^{{-1}})'||: "
      f"{np.linalg.norm(var_results.sigma_u - reconstructed_ab):.2e}")

### 4.1 Comparing Cholesky vs AB Model IRFs

Since the AB model we specified above has the same triangular structure as Cholesky
(but with an additional zero restriction: unemployment does not respond to inflation
or fed_funds contemporaneously), the IRFs will differ slightly.

In [ ]:
# Compute AB model IRFs
irf_ab = results_ab.irf(periods=20)

# Compare Cholesky vs AB for the monetary policy shock
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for i, (ax, name, title) in enumerate(zip(axes.flat, var_names, titles)):
    ax.plot(horizons, irf_cholesky[:, i, monetary_shock_idx],
            "b-", linewidth=1.8, label="Cholesky")
    ax.plot(horizons, irf_ab[:, i, monetary_shock_idx],
            "r--", linewidth=1.8, label="AB Model")
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_title(f"Response of {name}", fontsize=10)
    ax.set_xlabel("Quarters")
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.legend(fontsize=9)

plt.suptitle("Monetary Shock: Cholesky vs AB Model", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "svar_cholesky_vs_ab.png"), bbox_inches="tight")
plt.show()

### 4.2 Structural Shocks

One key output of SVAR estimation is the **recovered structural shocks**:

$$\hat{\varepsilon}_t = (B_0^{-1})^{-1} \hat{u}_t$$

These should be approximately i.i.d. and uncorrelated across equations if the model is
correctly identified.

In [ ]:
# Examine structural shocks
structural_shocks = results_cholesky.structural_shocks
print(f"Structural shocks shape: {structural_shocks.shape}")

# Verify they are uncorrelated
shock_corr = np.corrcoef(structural_shocks.T)
print(f"\nCorrelation of structural shocks (should be ~identity):")
print(pd.DataFrame(shock_corr.round(4), index=shock_names, columns=shock_names))

# Plot structural shocks time series
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for i, (ax, name) in enumerate(zip(axes, shock_names)):
    ax.plot(structural_shocks[:, i], color="steelblue", linewidth=0.8)
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_ylabel(name, fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Observation")
plt.suptitle("Recovered Structural Shocks (Cholesky)", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Validation with Synthetic Data

To verify our SVAR estimation, let us generate data from a **known structural process**
and check whether `chronobox` recovers the true structural impact matrix.

In [ ]:
from data_generators import generate_monetary_policy

# Generate synthetic monetary policy data with known DGP
synth_data, true_params = generate_monetary_policy(n=500, seed=42)
synth_names = true_params["variable_names"]

print(f"True structural impact matrix (B0_inv):")
print(pd.DataFrame(true_params["B0_inv"].round(4), 
                    index=synth_names, columns=synth_names))
print(f"\nNote: B0_inv is lower triangular => Cholesky should recover it")

# Fit VAR and SVAR on synthetic data
var_synth = VAR(lags=2, trend="c")
var_synth_res = var_synth.fit(synth_data)

svar_synth = SVAR(var_synth_res, method="cholesky")
res_synth = svar_synth.fit()

print(f"\nEstimated B0_inv:")
print(pd.DataFrame(res_synth.A0_inv.round(4), index=synth_names, columns=synth_names))

# Compare true vs estimated IRFs
irf_true_synth = np.zeros((21, 3, 3))
# Compute true IRFs from true parameters
Phi = [np.eye(3)]
A_list = [true_params["A1"], true_params["A2"]]
for h in range(1, 21):
    phi_h = np.zeros((3, 3))
    for j, A in enumerate(A_list):
        if h - j - 1 >= 0:
            phi_h += A @ Phi[h - j - 1]
    Phi.append(phi_h)
for h in range(21):
    irf_true_synth[h] = Phi[h] @ true_params["B0_inv"]

irf_est_synth = res_synth.irf(periods=20)

fig = plot_irf_comparison(
    irf_true_synth, irf_est_synth,
    variable_names=synth_names,
    shock_names=[f"{n} shock" for n in synth_names],
    title="Synthetic Data: True vs Estimated Structural IRF (Cholesky)",
)
plt.savefig(os.path.join("..", "outputs", "svar_validation_cholesky.png"), bbox_inches="tight")
plt.show()

## Exercicio

### Exercicio 1: Cholesky com ordering alternativo

Reordene as variaveis como `[inflation, gdp, fed_funds, unemployment]` — colocando
inflacao como a variavel mais exogena. Isso reflete uma visao onde a inflacao e
determinada por fatores de custo (cost-push) e nao responde ao ciclo no curto prazo.

1. Ajuste o VAR(4) com a nova ordenacao
2. Compute a IRF estrutural via Cholesky
3. Compare a resposta do GDP a um choque monetario com a ordenacao original
4. Qual ordenacao voce considera mais plausivel economicamente? Por que?

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

# Dica:
# alt_names = ["inflation", "gdp", "fed_funds", "unemployment"]
# endog_alt = df[alt_names].values
# var_alt = VAR(lags=4, trend="c")
# ...

### Exercicio 2: AB Model com restricoes nao-recursivas

Implemente um modelo AB onde a inflacao responde contemporaneamente tanto ao GDP
quanto ao fed_funds (canal de expectativas), mas o GDP nao responde a nenhuma outra
variavel contemporaneamente.

Restricoes:
- A[0,:] = [1, 0, 0, 0] (GDP exogeno)
- A[1,:] = [free, 1, free, 0] (inflacao responde a GDP e fed_funds)
- A[2,:] = [free, free, 1, 0] (Taylor rule: fed_funds responde a GDP e inflacao)
- A[3,:] = [free, 0, free, 1] (unemp responde a GDP e fed_funds)
- B = diagonal

Conte os parametros livres e verifique se o modelo e exatamente identificado.

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

# Dica:
# A_ex2 = np.array([
#     [1.0,     0.0,     0.0,     0.0],
#     [np.nan,  1.0,     np.nan,  0.0],
#     [np.nan,  np.nan,  1.0,     0.0],
#     [np.nan,  0.0,     np.nan,  1.0],
# ])
# B_ex2 = np.diag([np.nan]*4)
# svar_ex2 = SVAR(var_results, method="ab", a_mat=A_ex2, b_mat=B_ex2)
# ...

---

## Resumo

Neste notebook aprendemos:

1. **Forma reduzida vs forma estrutural**: residuos do VAR ($u_t$) sao misturas de
   choques estruturais ($\varepsilon_t$) — precisamos de restricoes para separa-los

2. **Cholesky (recursiva)**: impoe uma cadeia causal via ordenamento das variaveis.
   Simples, mas os resultados dependem da ordenacao escolhida

3. **Modelo AB**: permite restricoes mais flexiveis que nao precisam ser triangulares.
   Resolve via MLE. Requer que o numero de parametros livres seja $\leq K(K+1)/2$

4. **IRF estrutural**: a funcao resposta ao impulso tem interpretacao economica —
   cada choque corresponde a uma fonte especifica de variacao

5. **FEVD estrutural**: decomposicao da variancia do erro de previsao em contribuicoes
   de cada choque estrutural

No proximo notebook, veremos esquemas de identificacao baseados em **restricoes de longo
prazo** (Blanchard-Quah) e **restricoes de sinal** (Uhlig).